In [1]:
from transformers_sae import _autoreload

In [2]:
from transformers_sae.ops import load_checkpoint

my_sae = load_checkpoint(".checkpoints/gemma_2_2b/next_layer/layer_7_tokens_100001788.checkpoint").sae
assert my_sae is not None
my_sae.change_configured_device("cpu")
my_sae.eval()

SAE(
  (encoder): Encoder(
    (linear): Linear(in_features=2304, out_features=16384, bias=True)
    (activation): ModuleList(
      (0): BatchTopKActivationFunction()
    )
  )
  (decoder): Decoder(
    (linear): Linear(in_features=16384, out_features=2304, bias=True)
  )
)

In [3]:
from transformers_sae.sae_lens_wrapper import to_sae_lens

sae_lens_sae = to_sae_lens(my_sae, 7)

In [21]:
import torch
from sae_lens import HookedSAETransformer

model = None
model = HookedSAETransformer.from_pretrained_no_processing(
    "google/gemma-2-2b", dtype=torch.bfloat16, device="cpu"
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer


In [37]:
model.cfg.default_prepend_bos

True

In [48]:
with torch.no_grad(), torch.autocast(device_type="cpu", dtype=torch.bfloat16):
    tokens = tokenizer("This is a test", return_tensors="pt").input_ids[:, 0:1024]
    special_ids = torch.tensor(tokenizer.all_special_ids)
    special_token_indices = (
        (tokens[0].unsqueeze(-1) == special_ids).any(dim=-1).nonzero().squeeze(-1)
    )
    token_mask = torch.ones_like(tokens)
    token_mask.view(-1)[special_token_indices] = 0.0
    base_logits = model(tokens)

    pass_through_output = None

    def _cache_base_output(out, hook):
        global pass_through_output
        pass_through_output = out.view(out.shape[0] * out.shape[1], out.shape[2])[
            special_token_indices, :
        ]

    def _pass_through_special_tokens(out, hook):
        global pass_through_output
        out.view(out.shape[0] * out.shape[1], out.shape[2])[
            special_token_indices, :
        ] = pass_through_output
        return out

    sae_logits = model.run_with_hooks_with_saes(
        tokens,
        saes=[sae_lens_sae],
        fwd_hooks=[
            (
                f"{sae_lens_sae.cfg.metadata.hook_name}.hook_sae_input",
                _cache_base_output,
            ),
            (
                f"{sae_lens_sae.cfg.metadata.hook_name}.hook_sae_output",
                _pass_through_special_tokens,
            ),
        ],
    )
    print(
        torch.nn.functional.kl_div(
            sae_logits.log_softmax(-1),
            base_logits.log_softmax(-1),
            log_target=True,
            reduction="none",
        )
        .sum(dim=-1)[token_mask.bool()]
        .mean()
    )

tensor(0.0216)


In [6]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from transformers_sae.replacement_model import GemmaReplacement, make_replacement_model

model_id = "google/gemma-2-2b"

model = None
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cpu",
    dtype=torch.bfloat16,
    use_safetensors=True,
)
validation_dataset = load_dataset(
    "monology/pile-test-val",
    split="validation",
    revision="refs/convert/parquet",
    streaming=True,
    columns=["text"],
)
model = make_replacement_model(
    model,
    {},
    num_layers=model.config.num_hidden_layers,
    context_length=1024,  # model.config.max_position_embeddings,
    d_model=model.config.hidden_size,
    layer_path="model.layers",
    replacement_class=GemmaReplacement,
)
model.eval()
model.requires_grad_(False)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

GemmaReplacementInstance(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemm

In [ ]:
with torch.no_grad(), torch.autocast(device_type="cpu", dtype=torch.bfloat16):
    # tokens = tokenizer(
    #     next(validation_dataset.__iter__())["text"], return_tensors="pt"
    # ).input_ids[: model.context_length]
    tokens = tokenizer("This is a test", return_tensors="pt").input_ids[
        0, 0: model.context_length
    ]
    special_ids = torch.tensor(tokenizer.all_special_ids)
    special_token_indices = (
        (tokens[0].unsqueeze(-1) == special_ids).any(dim=-1).nonzero().squeeze(-1)
    )
    base_logits = model(tokens)[0]
    token_mask = torch.ones_like(tokens)
    token_mask.view(-1)[special_token_indices] = 0.0
    sae_logits = make_replacement_model(model, {7: my_sae})(
        tokens,
        token_mask=token_mask,
        pass_through_positions=special_token_indices,
    )[0]
    print(
        torch.nn.functional.kl_div(
            sae_logits.log_softmax(-1),
            base_logits.log_softmax(-1),
            log_target=True,
            reduction="none",
        )
        .sum(dim=-1)[token_mask.bool()]
        .mean()
    )

tensor(0.0266)
